<a href="https://colab.research.google.com/github/Anushadhirde/Urban-Heat-Island-Change-Detection/blob/main/Test_prithvi_load.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
!pip install terratorch

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.0/45.0 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 656.6/656.6 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 859.3/859.3 kB 36.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 174.8/174.8 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 849.3/849.3 kB 33.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 62.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 688.1/688.1 kB 32.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 30.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.4/72.4 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.6/165.6 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.8/168.8 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [1]:
"""
STEP: Load pretrained Prithvi with our custom 7 bands, and test it on one
real tile, to confirm the setup works before building the full training loop.

WHAT THIS DOES:
- Installs terratorch (IBM's official Prithvi fine-tuning toolkit)
- Builds a Prithvi-based segmentation model:
    - The backbone (the "understanding" part) loads PRETRAINED weights
    - Since our 7 bands (NDVI, NDBI, NDWI, Albedo, Elevation, Slope,
      Aspect) aren't Prithvi's original 6 reflectance bands, TerraTorch
      automatically gives those band's patch-embedding weights a fresh,
      randomly initialized start instead of pretrained values — this is
      the "band adaptation" we discussed.
    - A segmentation head on top is set up to output 4 classes, matching
      our ground truth classes (Cooling, No UHI, Weak UHI, Strong UHI)
- Loads ONE real tile + its mask from your data
- Runs a single forward pass to confirm the shapes and pipeline work

THIS SCRIPT DOES NOT TRAIN THE MODEL YET — it's a sanity check that
everything loads and connects correctly before we write the real training
loop (which will run this same setup over many epochs on all your tiles).

BEFORE YOU RUN:
- pip install terratorch  (run this first, in its own cell — it can take
  a few minutes and needs a restart of the Colab runtime afterward)
- Confirm the paths below match your setup.
"""

import numpy as np
import torch
import terratorch  # registers the Prithvi backbones
from terratorch.tasks import SemanticSegmentationTask

TILES_FOLDER = "/content/drive/MyDrive/DATASET/TILES"
INPUTS_FOLDER = f"{TILES_FOLDER}/inputs_normalized"
MASKS_FOLDER = f"{TILES_FOLDER}/masks"

# Our 7 bands, in the SAME order they were stacked (band 1 of the original
# stack, LST, is excluded — see tile_dataset.py comments for why).
BAND_NAMES = ["NDVI", "NDBI", "NDWI", "Albedo", "Elevation", "Slope", "Aspect"]
NUM_CLASSES = 4  # Cooling, No UHI, Weak UHI, Strong UHI

# Pick any tile that exists in your inputs_normalized folder to test with
SAMPLE_TILE_ID = "tile_00001"


def main():
    print("Building Prithvi segmentation model with custom 7-band input...")
    task = SemanticSegmentationTask(
        model_factory="EncoderDecoderFactory",
        model_args={
            "backbone": "prithvi_eo_v2_300_tl",   # pretrained Prithvi-EO-2.0 300M
            "backbone_pretrained": True,
            "backbone_bands": BAND_NAMES,          # our custom bands
            "backbone_num_frames": 1,              # single time-step per tile
            "num_classes": NUM_CLASSES,
            "necks": [{"name": "SelectIndices", "indices": [-1]},
                      {"name": "ReshapeTokensToImage"}],
            "decoder": "FCNDecoder",
        },
        loss="ce",  # cross-entropy, standard for multi-class segmentation
    )
    model = task.model
    model.eval()
    print("Model built successfully.\n")

    print(f"Loading sample tile: {SAMPLE_TILE_ID}")
    input_tile = np.load(f"{INPUTS_FOLDER}/{SAMPLE_TILE_ID}.npy")   # (7, 224, 224)
    mask_tile = np.load(f"{MASKS_FOLDER}/{SAMPLE_TILE_ID}.npy")     # (224, 224)
    print(f"  input_tile shape: {input_tile.shape}")
    print(f"  mask_tile shape:  {mask_tile.shape}")

    # any leftover NaNs (edge nodata) -> replace with 0 for this quick test;
    # the real training loop will handle this via loss masking instead
    input_tile = np.nan_to_num(input_tile, nan=0.0)

    # Prithvi expects shape: (batch, channels, time, height, width)
    input_tensor = torch.from_numpy(input_tile).float()
    input_tensor = input_tensor.unsqueeze(0).unsqueeze(2)  # add batch + time dims
    print(f"  input_tensor shape for model: {tuple(input_tensor.shape)}")

    print("\nRunning a single forward pass (no training yet)...")
    with torch.no_grad():
        output = model(input_tensor)

    print(f"Output shape: {tuple(output.output.shape)}")
    print(f"Expected roughly: (1, {NUM_CLASSES}, 224, 224) -> "
          f"(batch, class scores per pixel, height, width)")
    print("\nIf the output shape matches, the model loaded correctly and the "
          "7-band adaptation is working end to end.")


if __name__ == "__main__":
    main()

Building Prithvi segmentation model with custom 7-band input...


Prithvi_EO_V2_300M_TL.pt: reconstructing file:   0%|          |  0.00B / 1.33GB            

Prithvi_EO_V2_300M_TL.pt: downloading bytes:           |  0.00B            

Model built successfully.

Loading sample tile: tile_00001
  input_tile shape: (7, 224, 224)
  mask_tile shape:  (224, 224)
  input_tensor shape for model: (1, 7, 1, 224, 224)

Running a single forward pass (no training yet)...
Output shape: (1, 4, 224, 224)
Expected roughly: (1, 4, 224, 224) -> (batch, class scores per pixel, height, width)

If the output shape matches, the model loaded correctly and the 7-band adaptation is working end to end.
